# Module 04: Random Rotations & Why They're Magic

**Time**: ~35 minutes

**Prerequisites**: Module 03 (Walsh-Hadamard Transform)

In Module 03 we built the Walsh-Hadamard Transform and saw that it "mixes" coordinates -- it spreads information from one coordinate into many. But the WHT is **deterministic**: it always mixes the same way. That means some input vectors might be unlucky -- they could land on patterns where WHT doesn't spread well.

The fix is simple and powerful: **random sign flips** before the WHT. This gives us the **Randomized Hadamard Transform (RHT)**, which spreads information uniformly for *any* input vector. This is the key insight that makes TurboQuant work.

In [ ]:
import sys; sys.path.insert(0, "..")
from src.helpers import check_close, check_intuition, norm_preservation_test, section_header

import math
import torch
import torch.nn as nn
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Recap: WHT Spreads, but Deterministically

From Module 03, the Walsh-Hadamard Transform takes a vector and "mixes" all its coordinates together. Let's bring back our FWHT implementation:

In [ ]:
def fwht(x: torch.Tensor, normalize: bool = True) -> torch.Tensor:
    """Fast Walsh-Hadamard Transform (butterfly algorithm on last dim)."""
    d = x.shape[-1]
    assert d > 0 and (d & (d - 1)) == 0, f"Last dim must be power of 2, got {d}"

    result = x.clone()
    h = 1
    while h < d:
        result = result.view(*result.shape[:-1], -1, 2 * h)
        left = result[..., :h]
        right = result[..., h:]
        result = torch.cat([left + right, left - right], dim=-1)
        result = result.view(*x.shape)
        h *= 2

    if normalize:
        result = result / math.sqrt(d)
    return result

The problem: WHT is deterministic. Apply it to the same vector twice, you get the same output. Worse, some structured vectors don't get spread well. For example, the WHT of `[1, 1, 1, 1]` is `[2, 0, 0, 0]` (unnormalized) -- the information *concentrates* instead of spreading!

We need something that works for **any** input, not just lucky ones.

In [ ]:
# The all-ones vector is an eigenvector of WHT -- it doesn't get spread at all!
x_ones = torch.ones(8)
print("Input:       ", x_ones)
print("After WHT:   ", fwht(x_ones))
print("\nAll the energy stayed in coordinate 0. Not great for quantization.")

## 2. Random Sign Flips (Rademacher Vectors)

The fix is elegant: before applying WHT, multiply each coordinate by a random $\pm 1$ value. These random signs are called a **Rademacher vector**.

The key idea:
1. Generate a random vector of +1/-1 values
2. Multiply element-wise with the input: `x_flipped = x * signs`
3. Then apply WHT: `x_rotated = fwht(x_flipped)`

This combination is the **Randomized Hadamard Transform (RHT)**.

In [ ]:
section_header("Random Sign Flips", "The secret ingredient")

d = 8
gen = torch.Generator()
gen.manual_seed(42)

# Generate Rademacher signs: each entry is +1 or -1 with equal probability
signs = torch.randint(0, 2, (d,), generator=gen) * 2 - 1
print("Random signs:", signs)

# Now the all-ones vector gets scrambled before WHT
x_ones = torch.ones(d)
x_flipped = x_ones * signs
print("After sign flip:", x_flipped)

x_rotated = fwht(x_flipped)
print("After WHT:      ", x_rotated)
print("\nNow the energy is spread across all coordinates!")

## 3. Why Random Rotations Matter -- The Big Insight

Here's the core insight that makes TurboQuant work:

> **After the Randomized Hadamard Transform, the coordinates of ANY vector become nearly independent and identically distributed.**

This means:
- Take a vector like `[10, 0, 0, ..., 0]` where all information is in one coordinate
- After RHT: information is spread **uniformly** across all coordinates
- Take **any** vector: after RHT, coordinates look like draws from the same distribution
- Since coordinates are nearly i.i.d., we can **quantize each one separately** with a single scalar quantizer

Without this property, you'd need a different quantizer for every possible input shape. With it, one quantizer handles everything.

## 4. Visualizing the Spreading Effect

Let's see this in action. We'll take a "spiky" vector (one large coordinate, rest small) and show what happens before and after the RHT.

In [ ]:
section_header("Visualizing RHT", "From spiky to spread out")

d = 256

# Create a spiky vector: one large coordinate, rest near zero
x_spiky = torch.randn(d) * 0.01  # small noise in all coordinates
x_spiky[0] = 10.0                 # one big spike

# Apply RHT: sign flip then WHT
gen = torch.Generator()
gen.manual_seed(7)
signs = (torch.randint(0, 2, (d,), generator=gen) * 2 - 1).float()

x_flipped = x_spiky * signs
x_rotated = fwht(x_flipped)

print(f"Before RHT -- max: {x_spiky.abs().max():.2f}, std: {x_spiky.std():.4f}")
print(f"After  RHT -- max: {x_rotated.abs().max():.2f}, std: {x_rotated.std():.4f}")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Before RHT (spiky)", "After RHT (spread out)"),
)

fig.add_trace(
    go.Histogram(
        x=x_spiky.numpy(), nbinsx=50,
        name="Before", marker_color="#EF553B",
        opacity=0.8,
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Histogram(
        x=x_rotated.numpy(), nbinsx=50,
        name="After", marker_color="#636EFA",
        opacity=0.8,
    ),
    row=1, col=2,
)

fig.update_layout(
    title_text="RHT Turns a Spike into a Bell Curve",
    template="plotly_white",
    height=400,
    showlegend=False,
)
fig.update_xaxes(title_text="Coordinate value", row=1, col=1)
fig.update_xaxes(title_text="Coordinate value", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.show()

**What you should see**: The left histogram has one extreme outlier at 10 and everything else near zero. The right histogram is a nice bell curve centered at zero -- the energy has been spread evenly across all 256 coordinates.

This is exactly what we need for quantization: when all coordinates have similar magnitude, a single scalar quantizer can handle them all efficiently.

## 5. Build the RHT Class

Let's package this into a reusable class. We need:
- Padding to the next power of 2 (since FWHT requires it)
- A stored set of random signs (so the same seed always gives the same rotation)
- Forward: pad -> sign flip -> FWHT
- Inverse: FWHT -> sign flip -> truncate

In [ ]:
def next_power_of_2(n: int) -> int:
    """Return the smallest power of 2 >= n."""
    if n <= 0:
        return 1
    return 1 << (n - 1).bit_length()


class RandomizedHadamardTransform:
    """Randomized Hadamard Transform: random sign flips + FWHT.
    
    After applying RHT to a unit vector of dimension d, each coordinate
    is approximately N(0, 1/d) for large d. This is the key property
    that makes scalar quantization work well.
    """
    
    def __init__(self, dim: int, seed: int = 0):
        self.original_dim = dim
        self.padded_dim = next_power_of_2(dim)
        
        # Generate random Rademacher signs (+/-1)
        gen = torch.Generator()
        gen.manual_seed(seed)
        self.signs = (torch.randint(0, 2, (self.padded_dim,), generator=gen) * 2 - 1).float()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply RHT: pad, sign flip, FWHT."""
        # Pad to next power of 2 if needed
        if x.shape[-1] < self.padded_dim:
            padding = torch.zeros(*x.shape[:-1], self.padded_dim - x.shape[-1])
            x = torch.cat([x, padding], dim=-1)
        
        # Sign flip then Hadamard
        x = x * self.signs
        return fwht(x, normalize=True)
    
    def inverse(self, x: torch.Tensor) -> torch.Tensor:
        """Apply inverse RHT: FWHT, sign flip, truncate.
        
        Since the normalized Hadamard matrix is its own inverse (H = H^-1)
        and sign flips are self-inverse (s * s = 1), the inverse is just
        FWHT followed by sign flip, then truncate to original dimension.
        """
        x = fwht(x, normalize=True)
        x = x * self.signs
        return x[..., :self.original_dim]


# Quick test
rht = RandomizedHadamardTransform(dim=8, seed=42)
x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
y = rht.forward(x)
x_back = rht.inverse(y)
print(f"Input:        {x}")
print(f"After RHT:    {y}")
print(f"Roundtrip:    {x_back}")
print(f"Match: {torch.allclose(x, x_back, atol=1e-5)}")

## 6. Verify Properties

A good rotation should preserve norms (no information lost), be invertible (we can reconstruct), and be reproducible (same seed = same rotation).

In [ ]:
section_header("Property 1: Norm Preservation", "||RHT(x)|| == ||x||")

rht = RandomizedHadamardTransform(dim=64, seed=0)

# Test on several random vectors
vectors = torch.randn(100, 64)
norm_preservation_test(
    transform_fn=rht.forward,
    inverse_fn=rht.inverse,
    vectors=vectors,
    name="RHT",
)

In [ ]:
section_header("Property 2: Roundtrip Reconstruction", "inverse(forward(x)) == x")

rht = RandomizedHadamardTransform(dim=64, seed=0)
x = torch.randn(10, 64)
x_reconstructed = rht.inverse(rht.forward(x))
check_close(x, x_reconstructed, name="Roundtrip")

In [ ]:
section_header("Property 3: Different Seeds -> Different Rotations", "")

x = torch.randn(64)
rht_a = RandomizedHadamardTransform(dim=64, seed=0)
rht_b = RandomizedHadamardTransform(dim=64, seed=1)

y_a = rht_a.forward(x)
y_b = rht_b.forward(x)

diff = (y_a - y_b).abs().mean().item()
print(f"Mean abs difference between seed=0 and seed=1 outputs: {diff:.4f}")
print(f"Different rotations: {diff > 0.01}")

In [ ]:
section_header("Property 4: Same Seed -> Same Rotation", "Reproducibility")

rht_1 = RandomizedHadamardTransform(dim=64, seed=42)
rht_2 = RandomizedHadamardTransform(dim=64, seed=42)

x = torch.randn(64)
y_1 = rht_1.forward(x)
y_2 = rht_2.forward(x)

check_close(y_1, y_2, name="Same seed reproducibility")

## 7. The Key Experiment: Coordinate Independence

This is the experiment that really shows why RHT is magic for quantization.

If we generate random unit vectors and look at pairs of coordinates, they'll generally be correlated (they share the unit norm constraint). But after RHT, the coordinates become nearly **uncorrelated** -- they look like independent draws from the same distribution.

This means a single scalar quantizer can handle any coordinate, regardless of the input.

In [ ]:
section_header("Coordinate Independence Experiment", "The scatter plot test")

d = 256
n_vectors = 1000

# Generate random unit vectors
raw = torch.randn(n_vectors, d)
unit_vectors = raw / raw.norm(dim=-1, keepdim=True)

# Apply RHT
rht = RandomizedHadamardTransform(dim=d, seed=0)
rotated = rht.forward(unit_vectors)

# Pick coordinates 0 and 1
coord_0_before = unit_vectors[:, 0].numpy()
coord_1_before = unit_vectors[:, 1].numpy()
coord_0_after = rotated[:, 0].numpy()
coord_1_after = rotated[:, 1].numpy()

# Compute correlations
corr_before = np.corrcoef(coord_0_before, coord_1_before)[0, 1]
corr_after = np.corrcoef(coord_0_after, coord_1_after)[0, 1]
print(f"Correlation before RHT: {corr_before:.4f}")
print(f"Correlation after  RHT: {corr_after:.4f}")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Before RHT (corr={corr_before:.3f})",
        f"After RHT (corr={corr_after:.3f})",
    ),
)

fig.add_trace(
    go.Scattergl(
        x=coord_0_before, y=coord_1_before,
        mode="markers",
        marker=dict(size=3, color="#EF553B", opacity=0.5),
        name="Before",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scattergl(
        x=coord_0_after, y=coord_1_after,
        mode="markers",
        marker=dict(size=3, color="#636EFA", opacity=0.5),
        name="After",
    ),
    row=1, col=2,
)

fig.update_layout(
    title_text="Coordinate Independence: Before vs After RHT",
    template="plotly_white",
    height=450,
    width=900,
    showlegend=False,
)
fig.update_xaxes(title_text="Coordinate 0", row=1, col=1)
fig.update_yaxes(title_text="Coordinate 1", row=1, col=1)
fig.update_xaxes(title_text="Coordinate 0", row=1, col=2)
fig.update_yaxes(title_text="Coordinate 1", row=1, col=2)
fig.show()

**What you should see**: The right scatter plot should look like a circular cloud (uncorrelated coordinates), while the left may show some structure. After RHT, the coordinates behave like independent draws -- exactly the property we need to justify per-coordinate scalar quantization.

## 8. Connect to the Reference Code

Let's look at the actual `RandomizedHadamardTransform` from the TurboQuant reference implementation. It does the same thing as our class above, but with a few production-grade additions:

```python
class RandomizedHadamardTransform(nn.Module):
    def __init__(self, dim, seed=0):
        super().__init__()
        self.original_dim = dim
        self.padded_dim = _next_power_of_2(dim)
        gen = torch.Generator()
        gen.manual_seed(seed)
        signs = torch.randint(0, 2, (self.padded_dim,), generator=gen) * 2 - 1
        self.register_buffer("signs", signs.float())

    def forward(self, x):
        if x.shape[-1] < self.padded_dim:
            padding = torch.zeros(
                *x.shape[:-1], self.padded_dim - x.shape[-1],
                device=x.device, dtype=x.dtype,
            )
            x = torch.cat([x, padding], dim=-1)
        x = x * self.signs
        return fwht(x, normalize=True)

    def inverse(self, x):
        x = fwht(x, normalize=True)
        x = x * self.signs
        return x[..., :self.original_dim]
```

### Key differences from our simple class:

**`nn.Module` inheritance**: By inheriting from `torch.nn.Module`, the class integrates with PyTorch's ecosystem -- it can be part of a larger model, supports `.to(device)`, `.eval()`, state saving/loading, etc.

**`register_buffer("signs", ...)`**: This is crucial. A buffer is a tensor that:
- Moves with the module when you call `.cuda()` or `.to(device)` -- so if your model is on GPU, the signs automatically go to GPU too
- Gets saved/loaded with `state_dict()` -- so a trained model includes its rotation
- Is NOT treated as a parameter (no gradients) -- signs are fixed, not learned

**Device handling**: `device=x.device, dtype=x.dtype` on the padding tensor ensures everything stays on the same device. Our simple class only works on CPU.

**Padding for non-power-of-2**: Both versions pad to the next power of 2, since FWHT requires it. The inverse truncates back to the original dimension.

In [ ]:
section_header("Reference Implementation", "nn.Module version")

# Let's verify the reference code works the same way.
# Here's the nn.Module version:

class RHTModule(nn.Module):
    """Production-grade RHT as an nn.Module (mirrors reference code)."""
    signs: torch.Tensor  # type hint for the buffer
    
    def __init__(self, dim: int, seed: int = 0):
        super().__init__()
        self.original_dim = dim
        self.padded_dim = next_power_of_2(dim)
        gen = torch.Generator()
        gen.manual_seed(seed)
        signs = torch.randint(0, 2, (self.padded_dim,), generator=gen) * 2 - 1
        self.register_buffer("signs", signs.float())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] < self.padded_dim:
            padding = torch.zeros(
                *x.shape[:-1], self.padded_dim - x.shape[-1],
                device=x.device, dtype=x.dtype,
            )
            x = torch.cat([x, padding], dim=-1)
        x = x * self.signs
        return fwht(x, normalize=True)

    def inverse(self, x: torch.Tensor) -> torch.Tensor:
        x = fwht(x, normalize=True)
        x = x * self.signs
        return x[..., :self.original_dim]


# The module version produces identical results to our simple class
rht_simple = RandomizedHadamardTransform(dim=64, seed=0)
rht_module = RHTModule(dim=64, seed=0)

x = torch.randn(10, 64)
y_simple = rht_simple.forward(x)
y_module = rht_module.forward(x)
check_close(y_simple, y_module, name="Simple vs Module output")

# The module knows about its buffers
print(f"\nModule buffers: {list(rht_module.buffers())[0].shape}")
print(f"Module state_dict keys: {list(rht_module.state_dict().keys())}")

## 9. Exercise: Predict the Standard Deviation

Apply RHT to the vector `[1, 0, 0, ..., 0]` of dimension `d=64`.

**Question**: What should the standard deviation of the output coordinates be?

**Think about it**:
- The input has norm `||x|| = 1`
- RHT preserves the norm, so `||RHT(x)|| = 1`
- The energy (squared norm) is spread across all 64 coordinates
- If the coordinates are approximately i.i.d. with mean 0 and std $\sigma$, then $||y||^2 = \sum y_i^2 \approx d \cdot \sigma^2$
- So $\sigma \approx ||x|| / \sqrt{d} = 1/\sqrt{64} = 0.125$

In [ ]:
section_header("Exercise", "Predict the standard deviation")

d = 64
x = torch.zeros(d)
x[0] = 1.0  # unit vector along first axis

rht = RandomizedHadamardTransform(dim=d, seed=0)
y = rht.forward(x)

empirical_std = y.std().item()
theoretical_std = 1.0 / math.sqrt(d)

print(f"Theoretical std: 1/sqrt({d}) = {theoretical_std:.6f}")
print(f"Empirical std:   {empirical_std:.6f}")
print(f"Norm preserved:  ||x|| = {x.norm():.4f}, ||RHT(x)|| = {y.norm():.4f}")

check_intuition(
    result=empirical_std,
    expected=theoretical_std,
    hint="The std should be 1/sqrt(d) since norm is preserved and spread across d coordinates.",
    label="Standard deviation",
    tolerance=0.05,
)

In [ ]:
# Visualize: the output coordinates should all have magnitude ~1/sqrt(64)
fig = go.Figure()

fig.add_trace(go.Bar(
    x=list(range(d)),
    y=y.numpy(),
    name="RHT output",
    marker_color="#636EFA",
))

fig.add_hline(y=theoretical_std, line_dash="dash", line_color="red",
              annotation_text=f"+1/sqrt({d})")
fig.add_hline(y=-theoretical_std, line_dash="dash", line_color="red",
              annotation_text=f"-1/sqrt({d})")

fig.update_layout(
    title=f"RHT of [1, 0, 0, ..., 0] (d={d}): All coordinates near +/-1/sqrt(d)",
    xaxis_title="Coordinate index",
    yaxis_title="Value",
    template="plotly_white",
    height=400,
)
fig.show()

**What you should see**: All 64 bars have approximately the same magnitude ($\pm 0.125$). The sign of each bar is random (determined by the RHT's random signs), but the magnitude is exactly $1/\sqrt{64}$. This is because the first column of the Hadamard matrix has entries $\pm 1/\sqrt{d}$, and the sign flips just choose which sign each entry gets.

This is the core of TurboQuant: after RHT, every coordinate carries the same amount of information, so a single scalar quantizer works for all of them.

## Summary

| Concept | What it does | Why it matters |
|---------|-------------|----------------|
| Rademacher signs | Random +/-1 vector | Breaks deterministic structure of WHT |
| RHT = signs + WHT | Random orthogonal rotation | Works for ANY input, not just lucky ones |
| Norm preservation | `\|\|RHT(x)\|\| = \|\|x\|\|` | No information lost |
| Coordinate independence | After RHT, coordinates are ~i.i.d. | Can quantize each coordinate separately |
| `register_buffer` | PyTorch buffer for signs | Moves with model to GPU, saved in state_dict |

**Next up (Module 05)**: Now that we can make any vector's coordinates i.i.d., we'll design the optimal scalar quantizer for those coordinates -- the Lloyd-Max quantizer.